<a href="https://colab.research.google.com/github/EugeneOny1/-civil-infrastructure-monitoring---CS-PROJECT-II/blob/feature-branch/06_model_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 06 — Model Evaluation

This notebook evaluates the trained detector using:
- TensorFlow Object Detection API evaluation;
- test-set predictions where an exported SavedModel is available;
- class-wise detection statistics;
- a local Kenyan subset when present.

The purpose is to produce quantitative evidence for the final report.

**Do not report metrics until the model has been evaluated on data that was not used for training.**

In [1]:
# ---------------------------------------------------------
# Bootstrap TFOD environment for Notebook 06
# ---------------------------------------------------------

import os
import sys
import shutil
import subprocess
from pathlib import Path

TFOD_ENV = Path("/content/tfod_env")

if TFOD_ENV.exists():
    shutil.rmtree(TFOD_ENV)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "virtualenv"],
    check=True
)

subprocess.run(
    [sys.executable, "-m", "virtualenv", str(TFOD_ENV)],
    check=True
)

TFOD_PYTHON = str(TFOD_ENV / "bin" / "python")
TFOD_PIP = str(TFOD_ENV / "bin" / "pip")

packages = [
    "tensorflow[and-cuda]==2.15.0.post1",
    "tensorflow-estimator==2.15.0",
    "keras==2.15.0",
    "numpy==1.26.4",
    "protobuf==4.25.3",
    "grpcio-tools==1.59.3",
    "tf-models-official==2.15.0",
    "tensorflow-io==0.35.0",
    "tf-slim",
    "pycocotools",
    "lvis",
    "contextlib2",
    "avro-python3",
    "sacrebleu==2.2.0",
    "pyparsing==2.4.7",
]

subprocess.run(
    [TFOD_PIP, "install", "-q", "--upgrade", "pip", "setuptools", "wheel"],
    check=True
)

subprocess.run(
    [TFOD_PIP, "install", *packages],
    check=True
)

subprocess.run(
    [TFOD_PIP, "install", "--upgrade", "setuptools==68.2.2"],
    check=True
)

print("TFOD virtual environment created.")

TFOD virtual environment created.


In [2]:
# ---------------------------------------------------------
# Prepare TensorFlow Object Detection API for Notebook 06
# ---------------------------------------------------------

from pathlib import Path
import os
import shutil
import subprocess

MODELS_DIR = Path("/content/models")
RESEARCH_DIR = MODELS_DIR / "research"
OBJECT_DETECTION_DIR = RESEARCH_DIR / "object_detection"
SLIM_DIR = RESEARCH_DIR / "slim"
PROTOS_DIR = OBJECT_DETECTION_DIR / "protos"

# Clone TensorFlow Models only if it is not already available
if not OBJECT_DETECTION_DIR.exists():

    if MODELS_DIR.exists():
        shutil.rmtree(MODELS_DIR)

    print("Cloning TensorFlow Models repository...")

    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/tensorflow/models.git",
            str(MODELS_DIR),
        ],
        check=True,
    )

print("Research directory exists:", RESEARCH_DIR.exists())
print("Object Detection exists:", OBJECT_DETECTION_DIR.exists())
print("Slim exists:", SLIM_DIR.exists())

assert RESEARCH_DIR.exists(), (
    f"Missing: {RESEARCH_DIR}"
)

assert OBJECT_DETECTION_DIR.exists(), (
    f"Missing: {OBJECT_DETECTION_DIR}"
)

assert SLIM_DIR.exists(), (
    f"Missing: {SLIM_DIR}"
)

# Compile TensorFlow Object Detection protobuf files
proto_files = sorted(PROTOS_DIR.glob("*.proto"))

assert proto_files, (
    f"No .proto files found in {PROTOS_DIR}"
)

proto_command = [
    TFOD_PYTHON,
    "-m",
    "grpc_tools.protoc",
    f"-I{RESEARCH_DIR}",
    f"--python_out={RESEARCH_DIR}",
]

proto_command += [
    str(proto.relative_to(RESEARCH_DIR))
    for proto in proto_files
]

subprocess.run(
    proto_command,
    cwd=str(RESEARCH_DIR),
    check=True,
)

# Environment used by all TFOD subprocesses
TFOD_RUN_ENV = os.environ.copy()

TFOD_RUN_ENV["PYTHONPATH"] = (
    f"{RESEARCH_DIR}:{SLIM_DIR}"
)

TFOD_RUN_ENV["MPLBACKEND"] = "Agg"

print("TFOD source and protobufs ready.")
print("TFOD PYTHONPATH:", TFOD_RUN_ENV["PYTHONPATH"])

Cloning TensorFlow Models repository...
Research directory exists: True
Object Detection exists: True
Slim exists: True
TFOD source and protobufs ready.
TFOD PYTHONPATH: /content/models/research:/content/models/research/slim


In [5]:
# ---------------------------------------------------------
# Repair TF-Slim compatibility patch and verify TFOD
# ---------------------------------------------------------

from pathlib import Path
import subprocess

# ---------------------------------------------------------
# Locate tfexample_decoder.py WITHOUT importing tf_slim
# ---------------------------------------------------------

matches = list(
    Path("/content/tfod_env").glob(
        "lib/python*/site-packages/tf_slim/data/tfexample_decoder.py"
    )
)

assert matches, (
    "Could not find tfexample_decoder.py inside /content/tfod_env"
)

TFSLIM_DECODER = matches[0]

print("TF-Slim decoder:", TFSLIM_DECODER)

# ---------------------------------------------------------
# Repair the file
# ---------------------------------------------------------

text = TFSLIM_DECODER.read_text(
    encoding="utf-8"
)

lines = text.splitlines()

# Remove the tensorflow import inserted by the previous patch
lines = [
    line
    for line in lines
    if line.strip() != "import tensorflow as tf"
]

# Find all __future__ imports
future_indexes = [
    i
    for i, line in enumerate(lines)
    if line.strip().startswith("from __future__ import")
]

assert future_indexes, (
    "Could not find __future__ imports in tfexample_decoder.py"
)

# TensorFlow import must come AFTER all __future__ imports
insert_position = max(future_indexes) + 1

lines.insert(
    insert_position,
    "import tensorflow as tf"
)

text = "\n".join(lines) + "\n"

# ---------------------------------------------------------
# Apply TF-Slim compatibility replacements
# ---------------------------------------------------------

text = text.replace(
    "control_flow_ops.case(",
    "tf.case("
)

text = text.replace(
    "control_flow_ops.cond(",
    "tf.cond("
)

TFSLIM_DECODER.write_text(
    text,
    encoding="utf-8"
)

print("TF-Slim compatibility patch repaired.")

# ---------------------------------------------------------
# Verify TensorFlow Object Detection API
# ---------------------------------------------------------

check = subprocess.run(
    [
        TFOD_PYTHON,
        "-c",
        """
import tensorflow as tf
from object_detection.builders import model_builder

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
print("Object Detection imports: OK")
"""
    ],
    cwd=str(RESEARCH_DIR),
    env=TFOD_RUN_ENV,
    text=True,
    capture_output=True
)

print(check.stdout)

if check.returncode != 0:
    print(check.stderr)

assert check.returncode == 0, (
    "TFOD import verification failed."
)

print("Notebook 06 environment READY.")

TF-Slim decoder: /content/tfod_env/lib/python3.11/site-packages/tf_slim/data/tfexample_decoder.py
TF-Slim compatibility patch repaired.
TensorFlow: 2.15.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Object Detection imports: OK

Notebook 06 environment READY.


In [6]:
from pathlib import Path
import json
import subprocess
import sys
import os

import numpy as np
import pandas as pd


PROJECT_DIR = Path("/content/drive/MyDrive/CS PROJECT II")
DATA_DIR = PROJECT_DIR / "Data"
PROCESSED_DIR = DATA_DIR / "Processed Data"

TFRECORD_DIR = PROCESSED_DIR / "tfrecords"
SPLIT_DIR = PROCESSED_DIR / "annotations" / "splits"

TRAINING_DIR = (PROJECT_DIR / "training_checkpoints" / "ssd_mobilenet_v2_run_lr00125")
EXPORT_DIR = PROJECT_DIR / "exported_model"

TEST_RECORD = TFRECORD_DIR / "test.record"
TEST_MANIFEST = SPLIT_DIR / "test.jsonl"

MODEL_MAIN = Path("/content/models/research/object_detection/model_main_tf2.py")
TFOD_PYTHON = "/content/tfod_env/bin/python"
RESEARCH_DIR = Path("/content/models/research")
SLIM_DIR = RESEARCH_DIR / "slim"


LABEL_MAP_PATH = PROJECT_DIR / "configs" / "label_map.pbtxt"


print("Test record:", TEST_RECORD)

Test record: /content/drive/MyDrive/CS PROJECT II/Data/Processed Data/tfrecords/test.record


## 1. TensorFlow Object Detection API Evaluation

This uses the project's test record as the evaluation input.

If your `pipeline_runtime.config` points to validation data, create a test-specific evaluation config before running the official evaluation. Do not accidentally evaluate on the training set.

In [7]:
RUNTIME_CONFIG = PROJECT_DIR / "configs" / "pipeline_runtime.config"
TEST_EVAL_CONFIG = PROJECT_DIR / "configs" / "pipeline_test_eval.config"

if RUNTIME_CONFIG.exists():
    text = RUNTIME_CONFIG.read_text(encoding="utf-8")

    # Replace only the evaluation TFRecord path in a runtime copy.
    # This is intentionally simple and should be inspected if the config
    # contains multiple eval_input_reader blocks.
    old_candidates = [
        str(PROCESSED_DIR / "tfrecords" / "validation.record"),
        str(PROCESSED_DIR / "tfrecords" / "test.record"),
    ]

    replaced = False
    for old in old_candidates:
        if old in text:
            text = text.replace(old, str(TEST_RECORD))
            replaced = True

    if not replaced:
        print("WARNING: Could not automatically locate the validation TFRecord path.")
        print("Inspect pipeline_test_eval.config manually before using it.")

    TEST_EVAL_CONFIG.write_text(text, encoding="utf-8")
    print("Test evaluation config:", TEST_EVAL_CONFIG)
else:
    print("Missing runtime config. Complete Notebook 05 first.")

Test evaluation config: /content/drive/MyDrive/CS PROJECT II/configs/pipeline_test_eval.config


In [8]:
# ---------------------------------------------------------
# Run official TensorFlow Object Detection evaluation
# ---------------------------------------------------------

assert TEST_EVAL_CONFIG.exists(), (
    f"Missing test evaluation config: {TEST_EVAL_CONFIG}"
)

assert TRAINING_DIR.exists(), (
    f"Missing training directory: {TRAINING_DIR}"
)

assert Path(TFOD_PYTHON).exists(), (
    f"Missing TFOD Python environment: {TFOD_PYTHON}"
)

assert MODEL_MAIN.exists(), (
    f"Missing TensorFlow Object Detection evaluator: {MODEL_MAIN}"
)

command = [
    TFOD_PYTHON,
    str(MODEL_MAIN),
    f"--pipeline_config_path={TEST_EVAL_CONFIG}",
    f"--model_dir={TRAINING_DIR}",
    f"--checkpoint_dir={TRAINING_DIR}",
    "--eval_timeout=1",
    "--alsologtostderr",
]

print("Evaluation command:")
print(" ".join(command))

print("\nStarting evaluation...\n")

result = subprocess.run(
    command,
    cwd=str(RESEARCH_DIR),
    env=TFOD_RUN_ENV,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

print(result.stdout)

if result.returncode != 0:
    raise RuntimeError(
        "Model evaluation failed. "
        "The actual evaluator error is printed above."
    )

print("\nModel evaluation completed successfully.")

Evaluation command:
/content/tfod_env/bin/python /content/models/research/object_detection/model_main_tf2.py --pipeline_config_path=/content/drive/MyDrive/CS PROJECT II/configs/pipeline_test_eval.config --model_dir=/content/drive/MyDrive/CS PROJECT II/training_checkpoints/ssd_mobilenet_v2_run_lr00125 --checkpoint_dir=/content/drive/MyDrive/CS PROJECT II/training_checkpoints/ssd_mobilenet_v2_run_lr00125 --eval_timeout=1 --alsologtostderr

Starting evaluation...

2026-10-14 11:14:05.784172: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-14 11:14:05.833506: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-14 11:14:05.833702: E

## 2. Load Test Manifest

The manifest provides ground-truth boxes and classes for custom analysis.

In [9]:
def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

test_records = load_jsonl(TEST_MANIFEST)
print("Test images:", len(test_records))

print("Test datasets:")
print(pd.Series([r["dataset"] for r in test_records]).value_counts())

Test images: 355
Test datasets:
RDD2022        224
SDNET2018       89
Kenyan_Data     42
Name: count, dtype: int64


## 3. Optional Custom Evaluation Using Exported SavedModel

This section is enabled only when Notebook 07 has produced:

```text
exported_model/saved_model/
```

The code reads the model's `serving_default` signature and prints its actual output structure before calculating metrics.

This is deliberate because exported TensorFlow detection models can expose different tensor names depending on the exporter/model version.

In [10]:
SAVED_MODEL_DIR = EXPORT_DIR / "saved_model"

if SAVED_MODEL_DIR.exists():
    detect_fn = tf.saved_model.load(str(SAVED_MODEL_DIR))
    print("SavedModel loaded.")
    print(detect_fn.signatures["serving_default"].structured_outputs)
else:
    print("SavedModel not found yet.")
    print("Run Notebook 07 after training before using custom inference metrics.")

SavedModel not found yet.
Run Notebook 07 after training before using custom inference metrics.


## 4. IoU Matching Helper

A detection is considered a true positive when:
- predicted class matches the ground-truth class; and
- IoU is at least the chosen threshold.

This is a simplified analysis tool and should not replace the TensorFlow Object Detection API's official COCO evaluation.

In [11]:
def box_iou(box_a, box_b):
    # Boxes: [ymin, xmin, ymax, xmax]
    ymin_a, xmin_a, ymax_a, xmax_a = box_a
    ymin_b, xmin_b, ymax_b, xmax_b = box_b

    inter_ymin = max(ymin_a, ymin_b)
    inter_xmin = max(xmin_a, xmin_b)
    inter_ymax = min(ymax_a, ymax_b)
    inter_xmax = min(xmax_a, xmax_b)

    inter_h = max(0.0, inter_ymax - inter_ymin)
    inter_w = max(0.0, inter_xmax - inter_xmin)
    inter_area = inter_h * inter_w

    area_a = max(0.0, ymax_a - ymin_a) * max(0.0, xmax_a - xmin_a)
    area_b = max(0.0, ymax_b - ymin_b) * max(0.0, xmax_b - xmin_b)

    union = area_a + area_b - inter_area
    return inter_area / union if union > 0 else 0.0

## 5. Evaluation Notes

Record at least:
- mAP@0.5
- mAP@[0.5:0.95]
- precision
- recall
- F1 where calculated
- per-class results
- local Kenyan test results
- representative false positives and false negatives

Do not interpret high confidence as proof of engineering severity. Model confidence and relative visual severity are separate quantities.

In [12]:
print("Evaluation checklist")
print("✓ Official object-detection metrics")
print("✓ Per-class performance")
print("✓ Kenyan/local-domain performance")
print("✓ False positives")
print("✓ False negatives")
print("✓ Representative qualitative examples")

Evaluation checklist
✓ Official object-detection metrics
✓ Per-class performance
✓ Kenyan/local-domain performance
✓ False positives
✓ False negatives
✓ Representative qualitative examples
